# Modeling

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter

Preemptively set new Pandas option

In [ ]:
pd.options.mode.copy_on_write = True

Allow reloading of custom Python classes

In [ ]:
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

Putting in customer data

In [ ]:
# Initialize dict all data
sales_menu_customers_data = {}
for loc_id, df in sales_and_menu_data.items():

    # Prevent overwriting
    df = df.copy()

    # Keep the index, since merges don't keep it
    df.reset_index(inplace=True)

    # Double check customers are unique
    customers.dropna(subset=['customer_id'], inplace=True)
    customers.drop_duplicates(subset=['location_id', 'customer_id'], inplace=True)

    # Combine
    merged = pd.merge(df, customers, on=['location_id', 'customer_id'], how='left')

    # Reset the index back to datetimes
    merged.set_index('created_at', inplace=True, drop=False)

    # Save
    sales_menu_customers_data[loc_id] = merged

Add promo exposure indicator

In [ ]:
for loc_id, df in sales_menu_customers_data.items():
    df = df.tz_localize(None)
    df.loc[:,'promo'] = 0
    df.loc[:before_after_details.loc[loc_id,'cross_over_date'], 'promo'] = 1

Aggregated Data

In [ ]:
weekly_data_list = []
for loc_id, df in sales_menu_customers_data.items():

    df = df.copy()

    df['year'] = df['created_at'].dt.year
    df['month'] = df['created_at'].dt.month
    df['week'] = df['created_at'].dt.isocalendar().week
    df['day'] = df['created_at'].dt.isocalendar().day

    df['plant_based_sales'] = df.apply(lambda x: x['item_price'] if x['is_plant_based'] == 'Yes' else 0, axis=1)
    df['plant_based_quantity'] = df.apply(lambda x: x['item_quantity'] if x['is_plant_based'] == 'Yes' else 0, axis=1)

    aggregated_data = df.groupby(['location_id', 'year', 'month', 'week', 'day']).agg(
        transactions=('order_id', 'nunique'),
        all_sales=('item_price', 'sum'),
        plant_based_sales=('plant_based_sales', 'sum'),
        all_items=('item_quantity', 'sum'),
        plant_based_items=('plant_based_quantity', 'sum'),
        percent_female=('gender', lambda x: (x == 'female').mean() * 100),
    ).reset_index()

    aggregated_data['percent_items_plant_based'] = (aggregated_data['plant_based_items'] / aggregated_data['all_items']) * 100
    aggregated_data['percent_sales_plant_based'] = (aggregated_data['plant_based_sales'] / aggregated_data['all_sales']) * 100

    # Save
    weekly_data_list.append(aggregated_data)

# Make dataframe
weekly_data = pd.concat(weekly_data_list)

Putting in location data

In [ ]:
# Locations have data for cuisine, etc.
weekly_data_with_locations = pd.merge(weekly_data, locations, on='location_id', how='left')

In [ ]:
weekly_data_with_locations

View

In [ ]:
weekly_data_with_locations

In [ ]:
daily_data_with_promo_list = []
for loc_id in location_ids:
    promo_date = before_after_details.loc[loc_id,'cross_over_date'].isocalendar()
    year = promo_date.year
    week = promo_date.week
    day = promo_date.weekday

    # Treated: after the promo date
    treated = (weekly_data_with_locations.query(
        "location_id == @loc_id & ((year == @year & week == @week & day > @day) | "
        "(year == @year & week > @week) | (year > @year))", engine='python').assign(promo=1))
    
    # Untreated: before the promo date
    untreated = (weekly_data_with_locations.query(
        "location_id == @loc_id & ((year == @year & week == @week & day <= @day) | "
        "(year == @year & week < @week) | (year < @year))", engine='python').assign(promo=0))
        
    daily_data_with_promo_list.append(pd.concat([treated,untreated]))

daily_data_with_promo = pd.concat(daily_data_with_promo_list)

In [ ]:
daily_data_with_promo.columns

In [ ]:
agg_funcs = {
    'transactions': 'sum',  # Sum of transactions
    'all_sales': 'sum',  # Sum of all sales
    'plant_based_sales': 'sum',  # Sum of plant-based sales
    'all_items': 'sum',  # Sum of all items
    'plant_based_items': 'sum',  # Sum of plant-based items
    'percent_female': 'mean',
    'percent_items_plant_based': 'mean',  # Average of percent items plant-based
    'percent_sales_plant_based': 'mean'
}

# Group by the required keys and aggregate
weekly_data_with_promo = daily_data_with_promo.copy().groupby(['location_id', 'year', 'month', 'week', 'cuisine', 'city', 'state',
       'restaurant_type', 'pos_type', 'zip_code', 'neighborhood_age_5_under',
       'neighborhood_age_5_9', 'neighborhood_age_10_14',
       'neighborhood_age_15_19', 'neighborhood_age_20_24',
       'neighborhood_age_25_29', 'neighborhood_age_30_34',
       'neighborhood_age_35_39', 'neighborhood_age_40_44',
       'neighborhood_age_45_49', 'neighborhood_age_50_54',
       'neighborhood_age_55_59', 'neighborhood_age_60_64',
       'neighborhood_age_65_69', 'neighborhood_age_70_74',
       'neighborhood_age_75_79', 'neighborhood_age_80_84',
       'neighborhood_age_85_89', 'neighborhood_median_hh_income',
       'neighborhood_race_white', 'neighborhood_race_black',
       'neighborhood_race_americanIndian_alaskaNative',
       'neighborhood_race_asian',
       'neighborhood_race_nativeHawaiian_otherPacificIslander',
       'neighborhood_race_other', 'batch', 'promo']).agg(agg_funcs).reset_index()

In [ ]:
weekly_data_with_promo

Export

In [ ]:
daily_data_with_promo.to_parquet('data/daily_data.parquet')

In [ ]:
daily_data_with_promo

In [ ]:
daily_data_with_promo = pd.read_parquet('data/daily_data.parquet')

In [ ]:
daily_data_with_promo